# Proyecto 1 Etapa 1 (Aprendizaje de Maquina)

Alicia Robinson - 202321278

N Felipe Celis D - 202320636

Isabella Naranjo - 202321096

David Caro - 202222073

Version Final del mejor modelo obtenido

---
## 1.Imports y configuración

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import joblib
from scipy import stats

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

TRAIN_PATH = 'train.csv'
EVAL_PATH  = 'eval.csv'
OUTPUT_DIR = 'submission_fusion_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

import sklearn
print(f' Imports OK  |  scikit-learn {sklearn.__version__}')
print(f'   Output dir: {OUTPUT_DIR}/')

 Imports OK  |  scikit-learn 1.6.1
   Output dir: submission_fusion_v2/


---
## 2. Carga y exploración rápida


Se carga el archivo de entrenamiento y se revisan las propiedades básicas del dataset: cuántas filas tiene, qué rango de décadas cubre, cuántas clases distintas hay y si el dataset está balanceado.

El balance entre clases es relevante porque determina qué métrica usar. Si todas las décadas tienen aproximadamente la misma cantidad de ejemplos, accuracy es una métrica confiable. También se verifica que no haya valores nulos ni duplicados exactos que puedan distorsionar el entrenamiento.

In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)

print(f'Train shape     : {train_raw.shape}')
print(f'Columnas        : {list(train_raw.columns)}')
print(f'Décadas         : {train_raw["decade"].min()} – {train_raw["decade"].max()}')
print(f'Clases únicas   : {train_raw["decade"].nunique()}')

vc = train_raw['decade'].value_counts().sort_index()
print(f'\nBalance de clases:')
print(f'  Min por clase : {vc.min()}')
print(f'  Max por clase : {vc.max()}')
print(f'  Media         : {vc.mean():.0f}')
print(f'  Ratio max/min : {vc.max()/vc.min():.2f}x')
print(f'  → Dataset balanceado: accuracy es la métrica correcta ✓')

print(f'\nDuplicados exactos: {train_raw.duplicated().sum()}')
print(f'Valores nulos     : {train_raw.isnull().sum().sum()}')
train_raw.head(3)

Train shape     : (31403, 2)
Columnas        : ['text', 'decade']
Décadas         : 150 – 188
Clases únicas   : 39

Balance de clases:
  Min por clase : 754
  Max por clase : 848
  Media         : 805
  Ratio max/min : 1.12x
  → Dataset balanceado: accuracy es la métrica correcta ✓

Duplicados exactos: 34
Valores nulos     : 0


,text,decade
0,\nHonorarias ¡jubiladas. 57 \ndit.ad Pontem de...,164
1,"gone. Sus amigos , sus clientes, todo \ncuanto...",182
2,"Prefosen quemanera,e per qualesfolpechas deuan...",157


---
## 3.Preprocesamiento


La limpieza que se aplica es intencionalmente mínima. El objetivo es eliminar ruido estructural sin destruir las señales que permiten datar el texto.

Se normalizan los espacios en blanco y se eliminan párrafos que son boilerplate de Google Books (frases como "digitized by google" o "usage guidelines" que aparecen en libros escaneados y no tienen ningún valor temporal). También se descartan textos de menos de 10 caracteres y duplicados exactos.

Dejamos sin cambios: las mayúsculas, los acentos, los dígitos y la morfología de las palabras. En texto histórico en español estos elementos son señales temporales reales. Por lo cual no hicimos estas conversiones para no perder información útil para distinguir épocas.

La función preprocess_eval aplica exactamente la misma limpieza de texto sobre el set de evaluación, pero sin eliminar filas, porque en evaluación no podemos descartar ejemplos.

In [ ]:
# Patrones de boilerplate Google Books
_BP_PATTERNS = [
    'google books', 'google book search', r'books\.google\.com',
    'digitized by google', 'digitalizado por google',
    'scanned by google', 'carefully scanned by google',
    'keep it legal', 'maintain attribution',
    'non-commercial use', 'make non.commercial use',
    'haga un uso exclusivamente',
    'public domain for users', 'digitize public domain',
    'copyright infringement',
    'this is a digital copy',
    'usage guidelines',
    'new york public library', 'library of congress',
    'watermark',
]
BOILERPLATE_RE = re.compile('|'.join(_BP_PATTERNS), re.IGNORECASE)


def clean_text(text: str) -> str:
    """
    Limpieza mínima que preserva señales temporales.

    """
    if not isinstance(text, str):
        return ''
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


def preprocess_train(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['text'] = df['text'].astype(str).fillna('').apply(clean_text)

    # Eliminar textos vacíos o muy cortos
    before = len(df)
    df = df[df['text'].str.len() > 10].copy()
    print(f'  Textos muy cortos eliminados        : {before - len(df)}')

    # Eliminar boilerplate Google Books
    mask_keep = ~df['text'].str.contains(BOILERPLATE_RE, na=False)
    print(f'  Filas boilerplate eliminadas        : {(~mask_keep).sum()}')
    df = df[mask_keep].copy()

    # Eliminar duplicados exactos (texto y decada)
    before = len(df)
    df = df.drop_duplicates(subset=['text', 'decade']).copy()
    print(f'  Duplicados exactos eliminados       : {before - len(df)}')

    print(f'  Shape final                         : {df.shape}')
    return df


def preprocess_eval(df: pd.DataFrame) -> pd.DataFrame:
    """Misma limpieza de texto que train, SIN eliminar filas."""
    df = df.copy()
    df['text'] = df['text'].astype(str).fillna('').apply(clean_text)
    return df


print('Preprocesando train...')
train = preprocess_train(train_raw)

X_full = train['text'].values
y_full = train['decade'].astype(int).values

print(f'\nX_full: {X_full.shape}  |  Clases: {len(np.unique(y_full))}')

Preprocesando train...
  Textos muy cortos eliminados        : 0
  Filas boilerplate eliminadas        : 22
  Duplicados exactos eliminados       : 35
  Shape final                         : (31346, 2)

X_full: (31346,)  |  Clases: 39


---
## 4.FeatureUnion con pesos

## Representación del texto: cuatro vectorizadores en paralelo

Para convertir el texto en algo que un modelo pueda procesar, usamos TF-IDF, que básicamente le asigna un peso a cada secuencia de caracteres o palabras según qué tan frecuente es en un documento versus en el resto del corpus. No solo usamos un solo vectorizador sino cuatro, cada uno mirando el texto desde un ángulo distinto, y al final se concatenan todas esas representaciones.

Dos de ellos analizan secuencias cortas y largas de caracteres respetando los límites de palabra, otro hace lo mismo pero cruzando esos límites (lo que lo hace más tolerante a errores de escaneo), y el último trabaja con palabras completas y sus combinaciones de a dos y tres.

La razón de usar tantos es que en texto histórico en español la ortografía cambia mucho entre épocas. Una misma palabra puede escribirse "vuestro", "vueftro" o "vuezto" dependiendo del siglo, y si solo miramos palabras completas esas tres formas parecen cosas distintas. Los vectorizadores de caracteres capturan la señal temporal aunque otros aspectos cambie, por eso se les asigna más peso que al de palabras.

Los parámetros de filtrado (`min_df`, `max_df`, escala logarítmica) sirven para limpiar el vocabulario: se descartan términos que aparecen en muy pocos documentos (sin valor estadístico) y los que aparecen en casi todos (presentes en todas las épocas, por tanto no ayudan a distinguir nada).

In [ ]:
def build_feature_union_v2(
    char_wb_short_max : int = 120_000,
    char_wb_long_max  : int = 150_000,
    char_nowb_max     : int = 100_000,
    word_max          : int = 170_000,
) -> FeatureUnion:
    """
    FeatureUnion con transformer_weights diferenciados.


    """
    return FeatureUnion(
        transformer_list=[
            ('char_short', TfidfVectorizer(
                lowercase    = False,
                analyzer     = 'char_wb',
                ngram_range  = (1, 3),
                max_features = char_wb_short_max,
                min_df       = 2,
                max_df       = 0.95,
                sublinear_tf = True,
                strip_accents = 'unicode',
                dtype        = np.float32,
            )),
            ('char_long', TfidfVectorizer(
                lowercase    = False,
                analyzer     = 'char_wb',
                ngram_range  = (3, 6),
                max_features = char_wb_long_max,
                min_df       = 2,
                max_df       = 0.95,
                sublinear_tf = True,
                strip_accents = 'unicode',
                dtype        = np.float32,
            )),
            ('char_nowb', TfidfVectorizer(
                lowercase    = False,
                analyzer     = 'char',
                ngram_range  = (2, 6),
                max_features = char_nowb_max,
                min_df       = 2,
                max_df       = 0.95,
                sublinear_tf = True,
                strip_accents = 'unicode',
                dtype        = np.float32,
            )),
            ('word', TfidfVectorizer(
                lowercase     = False,
                analyzer      = 'word',
                ngram_range   = (1, 3),
                max_features  = word_max,
                min_df        = 2,
                max_df        = 0.95,
                sublinear_tf  = True,
                strip_accents = None,
                dtype         = np.float32,
            )),
        ],
        transformer_weights={
            'char_short': 1.5,
            'char_long' : 1.5,
            'char_nowb' : 1.3,
            'word'      : 1.0,
        },
    )


print(' FeatureUnion definido')
print('   char_wb×1.5, char×1.3, word×1.0')
print('   char nowb: ngram_range=(2,6) + strip_accents=unicode')

 FeatureUnion definido
   char_wb×1.5, char×1.3, word×1.0
   char nowb: ngram_range=(2,6) + strip_accents=unicode


---
## 5.Pipeline con VotingClassifier


Otra estrategia que usamos fue en lugar de apostar todo a un solo modelo, entrenamos tres clasificadores sobre las mismas features y dejamos que voten en conjunto. La idea es que cada uno tiene fortalezas distintas y falla en patrones distintos, así que combinados se compensan.

Los tres que usamos son Regresión Logística, una SVM lineal y un clasificador por descenso de gradiente estocástico. La Regresión Logística es el más estable de los tres y produce buenas estimaciones de probabilidad. La SVM lineal funciona especialmente bien con representaciones dispersas de alta dimensionalidad como las que produce TF-IDF. Y finalmente el SGD, entrenado con una función de pérdida que es menos sensible a valores extremos, aguanta mejor los textos con muchos errores de OCR.

El resultado final no se decide por mayoría de votos sino promediando las probabilidades que cada modelo asigna a cada clase. Esto importa porque no todos los votos pesan igual: si un clasificador está muy seguro de su respuesta (probabilidad alta) esa opinión arrastra más que una respuesta dudosa. En la práctica esta estrategia da mejores resultados que simplemente contar quién votó qué.

La SVM no produce probabilidades por defecto, así que se le aplica un paso de calibración para que su salida sea comparable con la de los otros dos y el promedio tenga sentido.

In [ ]:
def build_pipeline_v2(
    lr_C       : float = 5.0,
    svc_C      : float = 0.3,
    word_max   : int   = 170_000,
) -> Pipeline:
    """
    Pipeline v2: FeatureUnion pesado + VotingClassifier soft.

    """
    fu = build_feature_union_v2(word_max=word_max)

    lr = LogisticRegression(
        C            = lr_C,
        solver       = 'saga',
        max_iter     = 350,
        tol          = 2e-3,
        random_state = SEED,
        n_jobs       = 1,
    )

    # LinearSVC calibrado para obtener probabilidades compatibles con soft voting
    svc_base = LinearSVC(
        C            = svc_C,
        max_iter     = 2000,
        random_state = SEED,
    )
    svc = CalibratedClassifierCV(
        svc_base,
        cv           = 3,        # 3-fold interno para calibración
        method       = 'isotonic',
    )

    # SGD con modified_huber: pérdida cuadrática cerca del margen (más robusta que hinge)
    sgd = SGDClassifier(
        loss         = 'modified_huber',
        alpha        = 1e-4,
        max_iter     = 200,
        tol          = 1e-3,
        random_state = SEED,
        n_jobs       = 1,
        class_weight = None,
    )

    voting = VotingClassifier(
        estimators = [
            ('lr',  lr),
            ('svc', svc),
            ('sgd', sgd),
        ],
        voting  = 'soft',   # promedia probabilidades, no solo votos
        n_jobs  = 1,
    )

    return Pipeline([
        ('features', fu),
        ('clf',      voting),
    ])


print(' Pipeline v2 definido')
print('   VotingClassifier soft: LR(C=5) + SVC_calibrado(C=0.3) + SGD(modified_huber)')

 Pipeline v2 definido
   VotingClassifier soft: LR(C=5) + SVC_calibrado(C=0.3) + SGD(modified_huber)


---
## 6.GlobalSpecialistClassifier

Esta es la parte más importante del modelo. La idea surge de observar que el modelo global comete muchos errores dentro de rangos de décadas cercanas (por ejemplo, confunde 162 con 163), pero raramente confunde épocas distantes (no predice 155 cuando el texto es de 180).

Para aprovechar esto, se entrena primero un modelo global que predice sobre las 39 clases, y luego se entrena un modelo especialista por cada bloque de 5 décadas. Cuando el global predice que un texto pertenece, digamos, al bloque 160–164, el especialista de ese bloque toma la decisión final entre esas cinco opciones.

Los especialistas se entrenan únicamente con los ejemplos de su bloque, lo que les permite aprender los patrones especificos que distinguen décadas muy cercanas. Se les aplica una regularización para evitar overfitting dado que tienen menos datos.

Los pasos que sigue el modelo son:

1. El modelo global predice la década para cada texto.
2. Se identifica a qué bloque pertenece esa predicción.
3. El especialista del bloque re-predice entre las décadas del bloque.
4. La predicción del especialista es la respuesta final.


In [ ]:
class GlobalSpecialistClassifier(BaseEstimator, ClassifierMixin):
    """
    Clasificador jerárquico v2:
    1. Modelo global (VotingClassifier v2) → predicción inicial sobre 39 clases.
    2. Especialistas (VotingClassifier v2) → refinamiento focal por bloque.

    Parámetros
    ----------
    global_lr_C     : C para LogReg del modelo global.
    global_svc_C    : C para LinearSVC del modelo global.
    specialist_lr_C : C para LogReg de los especialistas.
    specialist_svc_C: C para LinearSVC de los especialistas.
    word_max        : Max features para vectorizador de palabras.
    blocks          : {nombre: [lista_de_décadas]}. None = bloques 8 por defecto.
    verbose         : Imprime progreso durante el entrenamiento.
    """

    # 8 bloques
    DEFAULT_BLOCKS = {
        'b150_154' : list(range(150, 155)),
        'b155_159' : list(range(155, 160)),
        'b160_164' : list(range(160, 165)),
        'b165_169' : list(range(165, 170)),
        'b170_174' : list(range(170, 175)),
        'b175_179' : list(range(175, 180)),
        'b180_184' : list(range(180, 185)),
        'b185_188' : list(range(185, 189)),
    }

    def __init__(
        self,
        global_lr_C      : float = 5.0,
        global_svc_C     : float = 0.3,
        specialist_lr_C  : float = 3.0,
        specialist_svc_C : float = 0.2,
        word_max         : int   = 170_000,
        blocks           : dict  = None,
        verbose          : bool  = True,
    ):
        self.global_lr_C      = global_lr_C
        self.global_svc_C     = global_svc_C
        self.specialist_lr_C  = specialist_lr_C
        self.specialist_svc_C = specialist_svc_C
        self.word_max         = word_max
        self.blocks           = blocks if blocks is not None else self.DEFAULT_BLOCKS
        self.verbose          = verbose

    def _to_series(self, X):
        return pd.Series(X).fillna('').astype(str).reset_index(drop=True)

    def _log(self, msg):
        if self.verbose:
            print(msg, flush=True)

    def fit(self, X, y):
        Xs = self._to_series(X)
        ys = pd.Series(y).reset_index(drop=True)

        # 1. Modelo global con VotingClassifier
        self._log('  [Global] Entrenando con VotingClassifier v2...')
        self.global_model_ = build_pipeline_v2(
            lr_C=self.global_lr_C,
            svc_C=self.global_svc_C,
            word_max=self.word_max,
        )
        self.global_model_.fit(Xs, ys)
        self._log('  [Global]  Listo')

        # 2. Especialistas por bloque
        self.specialists_ = {}
        for block_name, block_classes in self.blocks.items():
            mask    = ys.isin(block_classes)
            n_train = mask.sum()
            self._log(f'  [{block_name}] n={n_train} ejemplos... entrenando')

            if n_train < 20:
                self._log(f'      Muy pocos ejemplos, saltando')
                continue

            specialist = build_pipeline_v2(
                lr_C=self.specialist_lr_C,
                svc_C=self.specialist_svc_C,
                word_max=self.word_max,
            )
            specialist.fit(Xs[mask], ys[mask])
            self.specialists_[block_name] = {
                'classes' : set(block_classes),
                'model'   : specialist,
            }
            self._log(f'  [{block_name}]  Listo')

        self.classes_ = np.unique(ys)
        return self

    def predict(self, X):
        Xs = self._to_series(X)

        global_preds = pd.Series(
            self.global_model_.predict(Xs), index=Xs.index
        )
        final_preds = global_preds.copy()

        for info in self.specialists_.values():
            block_classes = info['classes']
            specialist    = info['model']

            # Solo re-clasifica donde el global predijo este bloque
            idx = global_preds[global_preds.isin(block_classes)].index
            if len(idx) == 0:
                continue

            final_preds.loc[idx] = specialist.predict(Xs.loc[idx])

        return final_preds.to_numpy()


print(' GlobalSpecialistClassifier definido')
print(f'   Bloques: {list(GlobalSpecialistClassifier.DEFAULT_BLOCKS.keys())}')

 GlobalSpecialistClassifier definido
   Bloques: ['b150_154', 'b155_159', 'b160_164', 'b165_169', 'b170_174', 'b175_179', 'b180_184', 'b185_188']


---
## 7.Validación cruzada

Se evalúa el pipeline base (sin los especialistas) mediante StratifiedKFold con 5 folds. El propósito es tener una estimación del score antes de generar el submission final.

No es necesario pero se utiliza para tener una estimación del score del modelo.(Se detuvo tempranamente por temas de tiempo para exportar el archivo del modelo)

In [ ]:
print('=' * 60)
print('Validación cruzada StratifiedKFold (5 folds)')
print('  Pipeline: FeatureUnion_v2 + VotingClassifier soft')
print('  (Sin especialistas — estimación conservadora)')
print('=' * 60)

cv_pipeline = build_pipeline_v2(lr_C=5.0, svc_C=0.3)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_scores = cross_val_score(
    cv_pipeline,
    X_full,
    y_full,
    cv      = skf,
    scoring = 'accuracy',
    n_jobs  = 1,
    verbose = 1,
)

print(f'\n CV Accuracy por fold: {[f"{s:.4f}" for s in cv_scores]}')
print(f'   Media  : {cv_scores.mean():.4f}')
print(f'   Std    : {cv_scores.std():.4f}')
print(f'\n   Estimación Kaggle (con especialistas): ~{cv_scores.mean() + 0.03:.4f}')

Validación cruzada StratifiedKFold (5 folds)
  Pipeline: FeatureUnion_v2 + VotingClassifier soft
  (Sin especialistas — estimación conservadora)


[Parallel(n_jobs=1)]: Done   0 out of   1 | elapsed:   29.6s remaining:   29.6s


KeyboardInterrupt: 

---
## 8.Entrenamiento del modelo final

Se instancia y entrena el `GlobalSpecialistClassifier` sobre el 100% de los datos de entrenamiento. Este paso incluye el entrenamiento del modelo global y de los 8 especialistas por bloque, cada uno con su propio FeatureUnion y VotingClassifier.



In [ ]:
print('=' * 60)
print('Entrenando modelo final v2 (Global + 8 Especialistas)')
print(f'  Datos: {len(X_full):,} ejemplos  |  39 clases')
print('  Clasificadores: LR(C=5) + SVC_cal(C=0.3) + SGD(m_huber)')
print('=' * 60)

model_final = GlobalSpecialistClassifier(
    global_lr_C      = 5.0,
    global_svc_C     = 0.3,
    specialist_lr_C  = 3.0,
    specialist_svc_C = 0.2,
    word_max         = 170_000,
    verbose          = True,
)
model_final.fit(X_full, y_full)

print('\n Modelo final v2 entrenado')

Entrenando modelo final v2 (Global + 8 Especialistas)
  Datos: 31,346 ejemplos  |  39 clases
  Clasificadores: LR(C=5) + SVC_cal(C=0.3) + SGD(m_huber)
  [Global] Entrenando con VotingClassifier v2...
  [Global]  Listo
  [b150_154] n=3967 ejemplos... entrenando
  [b150_154]  Listo
  [b155_159] n=4027 ejemplos... entrenando
  [b155_159]  Listo
  [b160_164] n=4069 ejemplos... entrenando
  [b160_164]  Listo
  [b165_169] n=4016 ejemplos... entrenando
  [b165_169]  Listo
  [b170_174] n=4098 ejemplos... entrenando
  [b170_174]  Listo
  [b175_179] n=3987 ejemplos... entrenando
  [b175_179]  Listo
  [b180_184] n=4016 ejemplos... entrenando
  [b180_184]  Listo
  [b185_188] n=3166 ejemplos... entrenando
  [b185_188]  Listo

 Modelo final v2 entrenado


---
## 9.Generación del submission

Se carga el archivo de evaluación, se aplica el mismo preprocesamiento de texto que al train, y se generan las predicciones con el modelo final.

Antes de guardar el CSV se corren cuatro validaciones de integridad: que el número de filas coincida con el set de evaluación, que no haya IDs duplicados, que no queden predicciones nulas y que todos los valores predichos caigan dentro del rango válido de décadas (150–188). Si alguna falla, el assert lanza un error explicativo.



In [ ]:
eval_raw = pd.read_csv(EVAL_PATH)
print(f'Eval shape: {eval_raw.shape}')

eval_df = preprocess_eval(eval_raw)
X_eval  = eval_df['text'].values

print('\nGenerando predicciones...', flush=True)
preds_main = model_final.predict(X_eval)

submission = pd.DataFrame({
    'id'    : eval_df['id'].values,
    'answer': preds_main,
})

# Validaciones de integridad
assert len(submission) == len(eval_df)
assert submission['id'].nunique() == len(submission)
assert submission['answer'].isnull().sum() == 0
assert submission['answer'].between(150, 188).all()

sub_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(sub_path, index=False)

print('\n Validaciones OK')
print(f' Guardado en: {sub_path}')
print(f'   Filas: {len(submission):,}')
print('\nDistribución de predicciones:')
print(submission['answer'].value_counts().sort_index().to_string())
print()
submission.head(10)

Eval shape: (3490, 2)

Generando predicciones...

 Validaciones OK
 Guardado en: submission_fusion_v2/submission.csv
   Filas: 3,490

Distribución de predicciones:
answer
150     88
151    115
152     99
153    104
154     99
155     84
156    109
157    112
158     98
159    111
160     96
161     70
162     62
163     90
164     64
165    109
166     65
167     79
168     93
169     77
170    101
171    113
172     77
173     82
174     83
175     88
176     65
177     64
178    110
179     79
180     80
181     74
182    104
183     76
184     89
185    107
186     91
187     81
188    102



,id,answer
0,0,173
1,1,188
2,2,150
3,3,171
4,4,153
5,5,174
6,6,165
7,7,160
8,8,159
9,9,152


---
## 10.Diagnóstico de errores por década

Para entender dónde falla el modelo se entrena una versión diagnóstica usando un split 85/15 del train (sin tocar el modelo final ya entrenado). Esto permite ver el accuracy individual por cada una de las 39 décadas.

El diagnóstico es útil para iteraciones futuras: si hay décadas con accuracy por debajo del 25%, son candidatas a recibir un especialista más fino o a ajustar los bloques. En particular, la zona 160–169 tiende a ser el cuello de botella porque esas décadas comparten mucho vocabulario y las diferencias ortográficas son muy pequeñas.

In [ ]:
print('Entrenando modelo de diagnóstico (85/15 split)...')
X_tr_d, X_val_d, y_tr_d, y_val_d = train_test_split(
    X_full, y_full, test_size=0.15, stratify=y_full, random_state=SEED
)

model_diag = GlobalSpecialistClassifier(
    global_lr_C=5.0, global_svc_C=0.3,
    specialist_lr_C=3.0, specialist_svc_C=0.2,
    word_max=170_000, verbose=False
)
model_diag.fit(X_tr_d, y_tr_d)

preds_diag  = model_diag.predict(X_val_d)
acc_overall = accuracy_score(y_val_d, preds_diag)
print(f'\n Accuracy global en val local: {acc_overall:.4f}')

rows = []
for dec in sorted(np.unique(y_val_d)):
    mask = y_val_d == dec
    acc_dec = (preds_diag[mask] == y_val_d[mask]).mean()
    rows.append({'Década': dec, 'N_val': mask.sum(), 'Accuracy': round(acc_dec, 3)})

res_df = pd.DataFrame(rows)
print('\nAccuracy por década:')
print(res_df.to_string(index=False))

hard = res_df[res_df['Accuracy'] < 0.25].sort_values('Accuracy')
print(f'\n Décadas con < 25% accuracy:')
if len(hard):
    print(hard.to_string(index=False))
else:
    print('  Ninguna ')

Entrenando modelo de diagnóstico (85/15 split)...


---
## 11.Guardar modelo

Se serializa el modelo entrenado con `joblib` y se verifica que la deserialización produce predicciones idénticas.

In [ ]:
# Guardar modelo principal
model_path = os.path.join(OUTPUT_DIR, 'model_fusion_v2.joblib')
joblib.dump(model_final, model_path)

# Verificación de integridad
loaded   = joblib.load(model_path)
verify   = loaded.predict(X_eval[:5])
original = model_final.predict(X_eval[:5])
assert np.array_equal(verify, original), ' Error de serialización'
print(f' Modelo guardado y verificado: {model_path}')




 Modelo guardado y verificado: submission_fusion_v2/model_fusion_v2.joblib
